# 02 · Cleaning

**Goal:** turn the raw table into the clean table used by phases 3 and 4. Every rule comes from an audit finding (notebook 01), fixes **sensor errors only**, and reports how many rows it touched.

| Rule | What |
|---|---|
| R1 | Parse the three timestamp formats, add `has_time` |
| R3 | Sentinel codes → NaN |
| R2 | Temperature °F → °C |
| R4 | Physical reactor key, drop `plant_country` |
| R5 | Flag rows with every sensor missing (kept) |
| R6 | No clipping of out-of-range values (deliberate non-rule) |
| R7 | No imputation here; it belongs in the phase 4 pipeline |
| R8 | Leakage columns kept, never used as features |

The order is R1 → R3 → R2 → R4 → R5: sentinels must be removed *before* the unit conversion, otherwise the 410 °F sentinel would turn into a plausible-looking 210 °C.

In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent / "src"))

import importlib
import pandas as pd
from fragrance import audit, cleaning, config

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 200)

## 1 · Run the rules
Load the raw table and run `cleaning.clean`, which returns the clean table and the report of rows touched per rule.

In [2]:
raw = audit.load_raw()
clean_df, report = cleaning.clean(raw)
report.style.format({"rows_touched": "{:,}", "share": "{:.1%}"})

,rule,description,rows_touched,share
0,R1,parse three timestamp formats,"25,107",9.7%
1,R3,sentinel codes -> NaN,"2,136",0.8%
2,R2,temperature °F -> °C,"1,487",0.6%
3,R4,"physical reactor key, drop plant_country","259,200",100.0%
4,R5,flag rows with all sensors missing,"5,344",2.1%


## 2 · Before / after checks
Each check confirms that one rule did what it promised, and nothing else: row count unchanged, no sentinel left, temperatures in a plausible °C range, out-of-range pH still present, 37 reactors, leakage columns kept.

In [3]:
sentinel_cols = [c for c in config.SENTINELS if c in clean_df.columns]
checks = {
    "same number of rows": len(clean_df) == len(raw),
    "every timestamp parsed": clean_df["event_time"].notna().all(),
    "no sentinel code left": not any(clean_df[c].isin(v).any() for c, v in config.SENTINELS.items() if c in clean_df.columns),
    "temperature within 75-100 °C (where present)": clean_df[config.TEMP_CELSIUS].dropna().between(75, 100).all(),
    "out-of-range pH kept (R6)": (clean_df["ph_level"] > 6).sum() == 4242,
    "37 physical reactors": clean_df["reactor"].nunique() == 37,
    "leakage columns kept (R8)": set(config.LEAKAGE_COLS) <= set(clean_df.columns),
    "5,344 empty records flagged (R5)": clean_df["all_sensors_missing"].sum() == 5344,
}
pd.Series(checks, name="passed")

same number of rows                             True
every timestamp parsed                          True
no sentinel code left                           True
temperature within 75-100 °C (where present)    True
out-of-range pH kept (R6)                       True
37 physical reactors                            True
leakage columns kept (R8)                       True
5,344 empty records flagged (R5)                True
Name: passed, dtype: bool

## 3 · Missing values after cleaning
Sentinels are now missing too, so each sensor's missing share rises slightly (≈ +0.2 points).

In [4]:
before = audit.missing_report(raw, config.SENSOR_COLS)["share_missing"].rename(index={config.TEMP_VALUE: config.TEMP_CELSIUS})
after = clean_df[config.CLEAN_SENSOR_COLS].isna().mean()
pd.DataFrame({"raw": before, "clean": after, "change": after - before}).style.format("{:.2%}")

,raw,clean,change
reaction_temperature_celsius,8.94%,9.14%,0.20%
vessel_pressure_bar,8.86%,9.07%,0.21%
refractive_index,8.86%,8.86%,0.00%
density_g_cm3,8.89%,9.11%,0.22%
ph_level,8.85%,9.06%,0.21%
mixing_torque_nm,8.83%,8.83%,0.00%
mass_flow_rate_kg_h,8.94%,8.94%,0.00%


## 4 · Save
Write `data/processed/batches_clean.parquet`.

In [5]:
cleaning.save_clean(clean_df)
print(f"saved {len(clean_df):,} rows x {clean_df.shape[1]} columns to {config.CLEAN_PARQUET.relative_to(config.ROOT)}")
clean_df.dtypes

saved 259,200 rows x 17 columns to data/processed/batches_clean.parquet


batch_id                                   str
plant_location                             str
production_line_id                         str
vessel_pressure_bar                    float64
refractive_index                       float64
density_g_cm3                          float64
ph_level                               float64
mixing_torque_nm                       float64
mass_flow_rate_kg_h                    float64
stability_months                       float64
purity_grade                               str
oxidation_risk_flag                       bool
event_time                      datetime64[ns]
has_time                                  bool
reaction_temperature_celsius           float64
reactor                                    str
all_sensors_missing                       bool
dtype: object

## Conclusions
- **Nothing was dropped:** 259,200 rows in, 259,200 rows out. Cleaning changed values, not batches.
- **What each rule touched:** R1 re-parsed 25,107 timestamps (9.7%), R3 cleared sentinels in 2,136 rows (0.8%), R2 converted 1,487 °F readings (0.6%), R4 added the reactor key to every row, R5 flagged 5,344 empty records (2.1%).
- **Temperature is now one Celsius column** (`reaction_temperature_celsius`), 79.0–93.1 °C, every value inside the 75–100 °C normal range. The °F readings were never anomalies.
- **Missing share rises by ~0.2 points** for the four sensors that had sentinels (to ~9.1%). The other three are unchanged. 41.4% of rows miss at least one sensor; filling them in is a phase 4 decision.
- **Signal preserved:** the 4,242 pH readings above 6 and every other plausible out-of-range value are untouched (R6).
- Output: `data/processed/batches_clean.parquet` (17 columns), regenerated by running this notebook, not committed.